In [ ]:
%%bash
nvidia-smi

In [ ]:
%%bash
apt-get update -qq && apt-get install -y -qq zstd || sudo apt-get update -qq && sudo apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh
ollama --version


In [ ]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama started")

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/")
print(response.status_code)
print(response.text)

In [ ]:
%%bash
ollama pull qwen3:8b
ollama list

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/v1/models")
print(response.status_code)
print(response.text)

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/v1/chat/completions",
    json={
        "model": "qwen3:8b",
        "messages": [
            {
                "role": "user",
                "content": "Say hello from the Kaggle GPU in one sentence."
            }
        ],
        "stream": False
    },
    timeout=600
)
print(response.status_code)
print(response.json())

In [ ]:
%%bash
wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
sudo dpkg -i cloudflared-linux-amd64.deb
cloudflared --version

In [ ]:
import subprocess, time, re, requests

NTFY_TOPIC = "kaggle-ollama-kiki-ecaf90ef"
LOG_PATH = "/tmp/cloudflared.log"
URL_PATH = "/tmp/kaggle_tunnel_url.txt"

# Start cloudflared in the background (non-blocking) so later cells can run.
log_file = open(LOG_PATH, "w")
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
print("cloudflared started, pid:", proc.pid)

# Poll the log for the public URL (usually appears in ~10-30s).
TUNNEL_URL = ""
pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for _ in range(60):
    time.sleep(2)
    with open(LOG_PATH, errors="ignore") as f:
        m = pattern.search(f.read())
    if m:
        TUNNEL_URL = m.group(0)
        break
    if proc.poll() is not None:
        raise RuntimeError("cloudflared exited early, see /tmp/cloudflared.log")
if not TUNNEL_URL:
    raise RuntimeError("Timed out waiting for trycloudflare URL in /tmp/cloudflared.log")

print("TUNNEL_URL:", TUNNEL_URL)
with open(URL_PATH, "w") as f:
    f.write(TUNNEL_URL)

# Post the URL to ntfy.sh so it can be retrieved externally while the kernel runs
# (the Kaggle API log is empty until the kernel finishes).
msg = f"Kaggle Ollama tunnel ready: {TUNNEL_URL}/v1  model: qwen3:8b"
try:
    ntfy_resp = requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", data=msg.encode(), timeout=30)
    print("ntfy post status:", ntfy_resp.status_code)
except Exception as e:
    print("ntfy post failed, tunnel still up:", e)
print("ntfy topic:", NTFY_TOPIC)


In [ ]:
import requests, time

# TUNNEL_URL comes from the previous cell; fall back to the saved file.
try:
    TUNNEL_URL
except NameError:
    with open("/tmp/kaggle_tunnel_url.txt") as f:
        TUNNEL_URL = f.read().strip()
print("Testing public URL:", TUNNEL_URL)

# DNS for a fresh trycloudflare hostname can take a few minutes to propagate.
# Retry for up to 5 minutes, and never let this test kill the tunnel.
models_response = None
for attempt in range(10):
    try:
        models_response = requests.get(f"{TUNNEL_URL}/v1/models", timeout=60)
        print("attempt", attempt+1, "status:", models_response.status_code)
        print(models_response.text[:500])
        break
    except Exception as e:
        print("attempt", attempt+1, "failed, waiting 30s:", e)
        time.sleep(30)
if models_response is None:
    print("WARNING: public /v1/models test did not succeed yet, tunnel may still come up - continuing to keep-alive")
else:
    try:
        chat_response = requests.post(
            f"{TUNNEL_URL}/v1/chat/completions",
            json={"model": "qwen3:8b", "messages": [{"role": "user", "content": "Say hello from the Kaggle GPU in one sentence."}], "stream": False},
            timeout=600
        )
        print(chat_response.status_code)
        print(str(chat_response.json())[:500])
    except Exception as e:
        print("WARNING: public chat test failed, continuing to keep-alive:", e)


In [ ]:
import time, datetime, requests

# Keep the tunnel alive for 4 hours so the endpoint can be used.
# Heartbeat every 10 minutes; re-checks the public endpoint each time.
for i in range(24):
    now = datetime.datetime.now().isoformat(timespec="seconds")
    try:
        r = requests.get(f"{TUNNEL_URL}/v1/models", timeout=30)
        print(f"[{now}] heartbeat {i+1}/24 - /v1/models -> {r.status_code} - tunnel alive: {TUNNEL_URL}")
    except Exception as e:
        print(f"[{now}] heartbeat {i+1}/24 - check failed: {e}")
    if i < 23:
        time.sleep(600)
print("Keep-alive finished (4 hours). Shut the session down to save GPU quota.")
